In [10]:
import sys
!{sys.executable} -m pip install snowflake-connector-python

In [15]:
import snowflake.connector

In [16]:
from snowflake.connector.pandas_tools import write_pandas

In [17]:
import pandas as pd

In [18]:
fact_transactions = pd.read_excel('All_Transactions.xlsx')

In [20]:
dim_users = pd.read_excel('All_Users.xlsx')

In [21]:
fact_transactions.head()

,Transaction_ID,Amount,User_ID,Service,Service Type,Payment_Status,Reason,Date
0,RCG_0C338474B366,926.59,PP0021371,Recharge_Bills,FASTag Recharge,Successful,Successful,2024-06-09
1,RCG_6B3B86B07A76,1211.64,PP0002388,Recharge_Bills,DTH,Successful,Successful,2024-08-04
2,RCG_767822392A0E,746.27,PP1101831,Recharge_Bills,Cable TV,Successful,Successful,2024-02-19
3,RCG_527E6AC74B11,1319.89,PP0033099,Recharge_Bills,Mobile Recharge,Successful,Successful,2024-12-22
4,RCG_6B50A8C694E1,112.44,PP1059869,Recharge_Bills,Cable TV,Successful,Successful,2024-09-07


In [22]:
dim_users.head()

,User_ID,Name,Age,Join_Date
0,PP0000001,Holly Rivera,56,2025-06-17
1,PP0000002,Kevin Lopez,46,2023-12-10
2,PP0000003,Douglas Roberts,32,2024-09-14
3,PP0000004,Walter Davila,60,2023-10-19
4,PP0000005,Grace Blake,25,2025-03-10


In [23]:
conn = snowflake.connector.connect(
    user="HARSHND",
    password="Harsh@@5757260",
    account="cm80021.ap-southeast-7.aws",
    warehouse="COMPUTE_WH",
    role="ACCOUNTADMIN"
)

cs = conn.cursor()
cs.execute("SELECT CURRENT_VERSION()")
print("Conneccted:", cs.fetchone())

Conneccted: ('10.32.102',)


In [24]:
cs.execute("CREATE DATABASE IF NOT EXISTS PHONEPE_ANALYTICS")
cs.execute("CREATE SCHEMA IF NOT EXISTS PHONEPE_ANALYTICS.PUBLIC")
print("Database and schema created")
cs.close()
conn.close()

Database and schema created


In [25]:
conn = snowflake.connector.connect(
    user="HARSHND",
    password="Harsh@@5757260",
    account="cm80021.ap-southeast-7.aws",
    warehouse="COMPUTE_WH",
    database="PHONEPE_ANALYTICS",
    schema="PUBLIC",
    role="ACCOUNTADMIN"
)
print("Connected with database and schema set.")

Connected with database and schema set.


In [26]:
fact_transactions.columns = [c.upper() for c in fact_transactions.columns]
dim_users.columns = [c.upper() for c in dim_users.columns]

# Loading fact_transaction data
success, nchunk, nrows, _ = write_pandas(
    conn, fact_transactions, "FACT_TRANSACTIONS",
    auto_create_table=True, overwrite=True
)

# Loading dim_users data
success, nchunk, nrows, _ = write_pandas(
    conn, dim_users, "DIM_USERS",
    auto_create_table=True, overwrite=True
)

conn.close()